# 🎯 ResNet CNN — CIFAR-100 Classification

Trains a **ResNet** model on **CIFAR-100** (100 classes, 60,000 images 32×32).

**Techniques used:**
- ResNet-18 / ResNet-50 (configurable) adapted for 32×32 input
- Label smoothing + Mixup augmentation
- Cosine annealing with warm restarts
- Grad-CAM visualization
- Top-1 / Top-5 accuracy tracking
- Confusion matrix for fine-grained classes

**CIFAR-100 structure:** 100 fine-grained classes grouped into 20 superclasses

In [ ]:
# ============================================================
# 1. INSTALL & IMPORTS
# ============================================================
!pip install grad-cam -q

import os
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as T
import torchvision.models as models

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

In [ ]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

# Architecture: 'resnet18', 'resnet34', 'resnet50'
ARCH         = 'resnet18'
PRETRAINED   = False     # True = use ImageNet weights (fine-tuning), False = train from scratch

BATCH_SIZE   = 128
EPOCHS       = 150
LR           = 0.1       # Initial LR (SGD with Cosine Annealing)
MOMENTUM     = 0.9
WEIGHT_DECAY = 5e-4
LABEL_SMOOTH = 0.1       # Label smoothing epsilon
MIXUP_ALPHA  = 0.4       # Mixup coefficient (0 = disabled)
DATA_DIR     = './data'
SAVE_PATH    = 'cifar100_resnet_best.pth'

NUM_CLASSES  = 100
print('Config OK')

In [ ]:
# ============================================================
# 3. DATA — CIFAR-100 with strong augmentation
# ============================================================

CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD  = (0.2675, 0.2565, 0.2761)

train_transform = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.AutoAugment(T.AutoAugmentPolicy.CIFAR10),  # CIFAR-adapted auto-augment
    T.ToTensor(),
    T.Normalize(CIFAR100_MEAN, CIFAR100_STD),
    T.RandomErasing(p=0.3, scale=(0.02, 0.1)),   # Cutout-style regularization
])

test_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

train_dataset = torchvision.datasets.CIFAR100(DATA_DIR, train=True,  download=True, transform=train_transform)
test_dataset  = torchvision.datasets.CIFAR100(DATA_DIR, train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train: {len(train_dataset)} | Test: {len(test_dataset)}')
print(f'Classes: {len(train_dataset.classes)}')

# CIFAR-100 class names for visualization
CLASS_NAMES = train_dataset.classes
print('Sample classes:', CLASS_NAMES[:10], '...')

In [ ]:
# ============================================================
# 4. VISUALIZE TRAINING DATA
# ============================================================

inv_norm = T.Normalize(
    mean=[-m/s for m, s in zip(CIFAR100_MEAN, CIFAR100_STD)],
    std=[1/s for s in CIFAR100_STD]
)

imgs, labels = next(iter(train_loader))
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for idx, ax in enumerate(axes.flat):
    if idx >= 32: break
    img = torch.clamp(inv_norm(imgs[idx]), 0, 1).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[labels[idx].item()], fontsize=7)
    ax.axis('off')
plt.suptitle('CIFAR-100 Training Samples', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cifar100_samples.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 5. MODEL — ResNet adapted for 32x32 CIFAR inputs
# ============================================================

def build_cifar_resnet(arch='resnet18', num_classes=100, pretrained=False):
    """
    Adapts ResNet for CIFAR-100:
    - Replace first 7x7 conv with 3x3 (CIFAR images are 32x32, not 224x224)
    - Remove first maxpool
    - Replace final FC for 100 classes
    """
    weight_map = {
        'resnet18': models.ResNet18_Weights.IMAGENET1K_V1,
        'resnet34': models.ResNet34_Weights.IMAGENET1K_V1,
        'resnet50': models.ResNet50_Weights.IMAGENET1K_V2,
    }
    builder = getattr(models, arch)
    weights = weight_map[arch] if pretrained else None
    model   = builder(weights=weights)

    # Adapt for 32x32 input: 7x7 conv, stride 2 → 3x3 conv, stride 1
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    # Remove maxpool (would shrink 32→16→8 too aggressively)
    model.maxpool = nn.Identity()

    # Replace final classifier
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )

    return model


model = build_cifar_resnet(ARCH, num_classes=NUM_CLASSES, pretrained=PRETRAINED).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Architecture: {ARCH} (CIFAR-adapted)')
print(f'Total params:     {total_params:,}')
print(f'Trainable params: {trainable_params:,}')

# Verify output shape
with torch.no_grad():
    dummy = torch.randn(4, 3, 32, 32).to(device)
    out   = model(dummy)
    print(f'Input: {dummy.shape} → Output: {out.shape}')

In [ ]:
# ============================================================
# 6. MIXUP AUGMENTATION
# ============================================================

def mixup_data(x, y, alpha=0.4):
    """Returns mixed inputs, pairs of targets, and lambda."""
    if alpha <= 0:
        return x, y, y, 1.0
    lam   = np.random.beta(alpha, alpha)
    idx   = torch.randperm(x.size(0)).to(x.device)
    x_mix = lam * x + (1 - lam) * x[idx]
    y_a, y_b = y, y[idx]
    return x_mix, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


print('Mixup functions ready')

In [ ]:
# ============================================================
# 7. LOSS, OPTIMIZER & SCHEDULER
# ============================================================

# Label smoothing cross-entropy
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)

# SGD with Nesterov momentum (standard for CIFAR training)
optimizer = optim.SGD(
    model.parameters(),
    lr=LR,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
    nesterov=True
)

# Warmup + Cosine annealing
warmup_epochs = 5
def lr_lambda(epoch):
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / (EPOCHS - warmup_epochs)
    return 0.5 * (1 + np.cos(np.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print(f'Optimizer: SGD | LR: {LR} | Momentum: {MOMENTUM} | WD: {WEIGHT_DECAY}')
print(f'Scheduler: Warmup ({warmup_epochs} epochs) + Cosine Annealing')

In [ ]:
# ============================================================
# 8. TRAINING LOOP
# ============================================================

def accuracy(output, target, topk=(1, 5)):
    with torch.no_grad():
        maxk = max(topk)
        batch_size = target.size(0)
        _, pred = output.topk(maxk, dim=1, largest=True, sorted=True)
        pred    = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))
        res = []
        for k in topk:
            c = correct[:k].reshape(-1).float().sum(0)
            res.append((c / batch_size * 100).item())
        return res


history = {'train_loss': [], 'train_acc1': [], 'val_loss': [], 'val_acc1': [], 'val_acc5': [], 'lr': []}
best_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    # ---- TRAIN ----
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0

    for imgs, labels in tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} [Train]', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)

        # Mixup
        imgs, ya, yb, lam = mixup_data(imgs, labels, alpha=MIXUP_ALPHA)

        optimizer.zero_grad()
        logits = model(imgs)
        loss   = mixup_criterion(criterion, logits, ya, yb, lam)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        t_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        t_correct += (preds == ya).sum().item()  # approximate (mixup)
        t_total   += imgs.size(0)

    scheduler.step()

    # ---- VALIDATE ----
    model.eval()
    v_loss, v_acc1_sum, v_acc5_sum, v_total = 0.0, 0.0, 0.0, 0

    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            v_loss += criterion(logits, labels).item() * imgs.size(0)
            acc1, acc5 = accuracy(logits, labels, topk=(1, 5))
            v_acc1_sum += acc1 * imgs.size(0)
            v_acc5_sum += acc5 * imgs.size(0)
            v_total    += imgs.size(0)

    avg_train_loss = t_loss   / t_total
    avg_train_acc  = 100 * t_correct / t_total
    avg_val_loss   = v_loss   / v_total
    avg_val_acc1   = v_acc1_sum / v_total
    avg_val_acc5   = v_acc5_sum / v_total
    current_lr     = optimizer.param_groups[0]['lr']

    history['train_loss'].append(avg_train_loss)
    history['train_acc1'].append(avg_train_acc)
    history['val_loss'].append(avg_val_loss)
    history['val_acc1'].append(avg_val_acc1)
    history['val_acc5'].append(avg_val_acc5)
    history['lr'].append(current_lr)

    if avg_val_acc1 > best_acc:
        best_acc = avg_val_acc1
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc1': best_acc,
            'val_acc5': avg_val_acc5,
        }, SAVE_PATH)

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:03d} | '
              f'Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.2f}% | '
              f'Val Loss: {avg_val_loss:.4f} | Val Top-1: {avg_val_acc1:.2f}% | '
              f'Val Top-5: {avg_val_acc5:.2f}% | LR: {current_lr:.6f}')

print(f'\n✅ Best Top-1 Accuracy: {best_acc:.2f}%')

In [ ]:
# ============================================================
# 9. TRAINING CURVES
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train', color='steelblue')
axes[0].plot(history['val_loss'],   label='Val',   color='coral')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history['train_acc1'], label='Train Top-1', color='steelblue')
axes[1].plot(history['val_acc1'],   label='Val Top-1',   color='coral')
axes[1].plot(history['val_acc5'],   label='Val Top-5',   color='green', linestyle='--')
axes[1].set_title('Accuracy (%)'); axes[1].set_xlabel('Epoch'); axes[1].legend()
axes[1].grid(True, alpha=0.3)

# LR Schedule
axes[2].plot(history['lr'], color='purple')
axes[2].set_title('Learning Rate Schedule'); axes[2].set_xlabel('Epoch')
axes[2].set_yscale('log'); axes[2].grid(True, alpha=0.3)

plt.suptitle(f'ResNet-{ARCH[-2:]} CIFAR-100 Training  |  Best Top-1: {best_acc:.2f}%',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cifar100_training_curves.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 10. EVALUATION ON TEST SET
# ============================================================

# Load best checkpoint
ckpt = torch.load(SAVE_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Loaded best model from epoch {ckpt["epoch"]} | Val Top-1: {ckpt["val_acc1"]:.2f}%')

model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc='Testing'):
        imgs   = imgs.to(device)
        logits = model(imgs)
        probs  = F.softmax(logits, dim=1).cpu()
        preds  = logits.argmax(dim=1).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())
        all_probs.append(probs)

all_probs  = torch.cat(all_probs, dim=0).numpy()
top1_acc   = np.mean(np.array(all_preds) == np.array(all_labels)) * 100
top5_correct = sum(
    1 for probs, label in zip(all_probs, all_labels)
    if label in np.argsort(probs)[-5:]
)
top5_acc = top5_correct / len(all_labels) * 100

print(f'\n📊 FINAL TEST RESULTS')
print(f'   Top-1 Accuracy: {top1_acc:.2f}%')
print(f'   Top-5 Accuracy: {top5_acc:.2f}%')

In [ ]:
# ============================================================
# 11. CONFUSION MATRIX (top-20 superclasses)
# ============================================================

# CIFAR-100 superclass mapping
SUPERCLASSES = [
    'aquatic mammals', 'fish', 'flowers', 'food containers', 'fruit & vegetables',
    'household electrical devices', 'household furniture', 'insects', 'large carnivores',
    'large man-made outdoor things', 'large natural outdoor scenes', 'large omnivores/herbivores',
    'medium-sized mammals', 'non-insect invertebrates', 'people', 'reptiles',
    'small mammals', 'trees', 'vehicles 1', 'vehicles 2'
]
FINE_TO_SUPER = [
    4,1,14,8,0,6,7,7,18,3,3,14,9,18,7,11,3,9,7,11,
    6,11,5,10,7,6,13,15,3,15,0,11,1,10,12,14,16,9,11,5,
    5,19,8,8,15,13,14,17,18,10,16,4,17,4,2,0,17,4,18,17,
    10,3,2,12,12,16,12,1,9,19,2,10,0,1,16,12,9,13,15,13,
    16,19,2,4,6,19,5,5,8,19,18,1,2,15,6,0,17,8,14,13
]

super_preds  = [FINE_TO_SUPER[p] for p in all_preds]
super_labels = [FINE_TO_SUPER[l] for l in all_labels]

cm_super = confusion_matrix(super_labels, super_preds)
cm_norm  = cm_super.astype(float) / cm_super.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=SUPERCLASSES, yticklabels=SUPERCLASSES,
            ax=ax, linewidths=0.5, vmin=0, vmax=1)
ax.set_xlabel('Predicted Superclass', fontsize=11)
ax.set_ylabel('True Superclass', fontsize=11)
ax.set_title('Normalized Confusion Matrix — CIFAR-100 Superclasses', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('confusion_matrix_superclasses.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 12. PER-CLASS ACCURACY — TOP 10 BEST & WORST
# ============================================================

per_class_acc = {}
for cls_idx in range(NUM_CLASSES):
    mask = np.array(all_labels) == cls_idx
    if mask.sum() > 0:
        per_class_acc[CLASS_NAMES[cls_idx]] = np.array(all_preds)[mask] == cls_idx
        per_class_acc[CLASS_NAMES[cls_idx]] = per_class_acc[CLASS_NAMES[cls_idx]].mean() * 100

sorted_acc = sorted(per_class_acc.items(), key=lambda x: x[1], reverse=True)
top10_best  = sorted_acc[:10]
top10_worst = sorted_acc[-10:]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

names, accs = zip(*top10_best)
axes[0].barh(names, accs, color='steelblue'); axes[0].set_xlim(0, 100)
axes[0].set_title('Top 10 Best Classes'); axes[0].set_xlabel('Accuracy (%)')
axes[0].invert_yaxis(); axes[0].grid(True, alpha=0.3, axis='x')
for i, v in enumerate(accs): axes[0].text(v+0.5, i, f'{v:.1f}%', va='center', fontsize=9)

names, accs = zip(*top10_worst)
axes[1].barh(names, accs, color='coral'); axes[1].set_xlim(0, 100)
axes[1].set_title('Top 10 Worst Classes'); axes[1].set_xlabel('Accuracy (%)')
axes[1].invert_yaxis(); axes[1].grid(True, alpha=0.3, axis='x')
for i, v in enumerate(accs): axes[1].text(v+0.5, i, f'{v:.1f}%', va='center', fontsize=9)

plt.suptitle('Per-Class Accuracy Analysis — CIFAR-100', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('per_class_accuracy.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 13. GRAD-CAM VISUALIZATION
# ============================================================
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# Target the last ResNet layer
target_layer = [model.layer4[-1]]
cam = GradCAM(model=model, target_layers=target_layer)

inv_norm_fn = T.Normalize(
    mean=[-m/s for m,s in zip(CIFAR100_MEAN, CIFAR100_STD)],
    std=[1/s for s in CIFAR100_STD]
)

n_vis = 16
vis_imgs, vis_labels = [], []
for imgs, labels in test_loader:
    vis_imgs.extend(imgs[:n_vis])
    vis_labels.extend(labels[:n_vis].tolist())
    if len(vis_imgs) >= n_vis: break
vis_imgs = vis_imgs[:n_vis]

fig, axes = plt.subplots(4, 8, figsize=(20, 10))
fig.suptitle('Grad-CAM Visualizations — CIFAR-100', fontsize=14, fontweight='bold')

for i in range(n_vis):
    img_tensor = vis_imgs[i].unsqueeze(0).to(device)
    label      = vis_labels[i]
    targets    = [ClassifierOutputTarget(label)]

    grayscale_cam = cam(input_tensor=img_tensor, targets=targets)
    grayscale_cam = grayscale_cam[0]

    img_rgb  = torch.clamp(inv_norm_fn(vis_imgs[i]), 0, 1).permute(1, 2, 0).numpy()
    # Upsample CAM to 32x32
    cam_vis  = show_cam_on_image(img_rgb, grayscale_cam, use_rgb=True)

    row, col = (i * 2) // 8, (i * 2) % 8
    axes[row, col].imshow(img_rgb)
    axes[row, col].set_title(CLASS_NAMES[label], fontsize=7)
    axes[row, col].axis('off')

    axes[row, col+1].imshow(cam_vis)
    axes[row, col+1].set_title('Grad-CAM', fontsize=7)
    axes[row, col+1].axis('off')

plt.tight_layout()
plt.savefig('gradcam.png', dpi=150)
plt.show()

In [ ]:
# ============================================================
# 14. INFERENCE ON CUSTOM IMAGE
# ============================================================

def predict_image(model, img_path_or_pil, top_k=5):
    """Predict top-k classes for a single image."""
    transform = T.Compose([
        T.Resize((32, 32)),
        T.ToTensor(),
        T.Normalize(CIFAR100_MEAN, CIFAR100_STD)
    ])
    if isinstance(img_path_or_pil, str):
        from PIL import Image
        img = Image.open(img_path_or_pil).convert('RGB')
    else:
        img = img_path_or_pil

    tensor = transform(img).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        probs  = F.softmax(logits, dim=1)[0]

    top_probs, top_idxs = probs.topk(top_k)
    print(f'\nTop-{top_k} Predictions:')
    for p, idx in zip(top_probs.tolist(), top_idxs.tolist()):
        print(f'  {CLASS_NAMES[idx]:30s}: {p*100:.2f}%')

    plt.figure(figsize=(6, 4))
    plt.barh([CLASS_NAMES[i] for i in top_idxs.tolist()[::-1]],
             [p*100 for p in top_probs.tolist()[::-1]], color='steelblue')
    plt.xlabel('Confidence (%)')
    plt.title(f'Top-{top_k} Predictions')
    plt.tight_layout()
    plt.show()
    return top_idxs.tolist(), top_probs.tolist()


# Example usage:
# predict_image(model, '/path/to/image.jpg')
print('predict_image() ready. Usage: predict_image(model, "path/to/image.jpg")')

## Summary

| Component | Details |
|-----------|--------|
| **Architecture** | ResNet-18 (CIFAR-adapted: 3×3 conv, no maxpool) |
| **Dataset** | CIFAR-100 (50K train / 10K test, 100 classes) |
| **Augmentation** | RandomCrop, HFlip, AutoAugment, RandomErasing, Mixup |
| **Loss** | CrossEntropy + Label Smoothing (ε=0.1) |
| **Optimizer** | SGD + Nesterov Momentum |
| **Scheduler** | Warmup (5ep) + Cosine Annealing |
| **Metrics** | Top-1, Top-5, Per-class Accuracy, Confusion Matrix |

**Expected performance (ResNet-18, scratch):**
- Top-1: ~75-78% | Top-5: ~92-94%

**Tips to improve:**
- Use `ARCH='resnet50'` for higher capacity
- Set `PRETRAINED=True` for fine-tuning from ImageNet
- Increase epochs to 200
- Try `MIXUP_ALPHA=0.8` or add CutMix